# MLP — Lo-Hi Benchmark (Drug Discovery)

Notebook per addestrare un **Multi-Layer Perceptron** sul benchmark Lo-Hi (Steshin, NeurIPS 2023),
seguendo la stessa metodologia nested CV usata per gli altri modelli della tesi (LR/KNN/DT/SVM/RF/GB):

- **Outer loop**: 3 fold pre-esistenti (split Tanimoto sim < 0.4) → test toccato una volta sola
- **Inner loop**: 2 fold (StratifiedKFold per Hi, KFold per Lo) per model selection
- **Retraining** su `train_i` con i best-params → valutazione su `test_i`
- **Aggregazione**: mean ± std sui 3 outer fold

A differenza degli altri modelli (sklearn-based) qui uso **PyTorch puro** per avere controllo
fine su:
- profondità (numero layer) e shape (piramidale / costante)
- regularization (L2 weight-decay, dropout, early stopping)
- ottimizzatore e scheduler
- multiple random init per stabilizzare la media (cfr. eterogeneità stocastica del training)

### Pipeline a due fasi

1. **Random search** (~60 config) sullo spazio largo → identifica zone promettenti
2. **Grid search raffinata** sulle zone migliori → fine tuning

### Note metodologiche (Micheli, ML 2025)

- **ReLU + He init** per gli hidden (gradient vanish mitigato sotto profondità)
- **Early stopping** + **weight decay (L2)** come regolarizzatori principali
- **Gradient clipping** per stabilità (cliff in cost surface)
- Scaling target solo per Lo (regression), non per Hi (classification)

> Una parte dello scheletro di training (early stopping con multiple inits, mediana sulle curve,
> normalizzazione lr/batch-size) è riadattata dalla mia pipeline ML-CUP25.


## 1. Configurazione

Variabili globali da cambiare per riusare il notebook su un altro dataset.
Il resto del codice è dataset-agnostic.

In [ ]:
# === Dataset config (cambia QUI per altri dataset) ===
DATASET = "drd2"
TASK    = "hi"          # "hi" oppure "lo"
MODEL_NAME = f"mlp_{DATASET}"   # naming coerente con il resto del progetto

# === Paths ===
import os
from pathlib import Path

# Assumendo notebook in <repo>/notebooks/mlp/  → root = parents[2]
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR     = PROJECT_ROOT / "data"     / TASK / DATASET
RESULTS_DIR  = PROJECT_ROOT / "results"  / TASK / DATASET / f"{MODEL_NAME}_ecfp4"
FEATURES_DIR = PROJECT_ROOT / "features"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Results dir:  {RESULTS_DIR}")


## 2. Imports & seeding

In [ ]:
import json
import copy
import time
import datetime
import warnings
from typing import Callable, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import (StratifiedKFold, KFold,
                                     ParameterGrid, ParameterSampler)
from sklearn.preprocessing  import StandardScaler
from sklearn.metrics        import average_precision_score
from scipy.stats            import spearmanr, loguniform, uniform, randint

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

warnings.filterwarnings("ignore", category=UserWarning)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 3. Featurization (ECFP4)

Per l'MLP uso **ECFP4 (Morgan radius=2, 2048 bit)** come fingerprint principale.
È quello più informativo sui task drug-discovery e quello che ha dato i risultati migliori
con i modelli lineari/kernel (cfr. tabelle SVM/LR del progetto).

> Si potrebbero esplorare anche MACCS (167 bit) e RDKit descriptors, ma a parità di MLP
> la dimensionalità maggiore di ECFP4 sfrutta meglio la capacità espressiva della rete.
> Lascio cache su `features/` per non rifare il calcolo a ogni fold.


In [ ]:
def smiles_to_ecfp4(smiles: str, n_bits: int = 2048, radius: int = 2) -> np.ndarray:
    """Single SMILES -> ECFP4 bit vector. Returns zeros if parsing fails."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits, dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    from rdkit.DataStructs import ConvertToNumpyArray
    ConvertToNumpyArray(fp, arr)
    return arr


def featurize_split(df: pd.DataFrame, smiles_col: str = "smiles") -> np.ndarray:
    """Vectorize a dataframe of SMILES into an (N, 2048) ECFP4 matrix."""
    return np.stack([smiles_to_ecfp4(s) for s in df[smiles_col].values])


def load_fold(fold_idx: int):
    """Load fold i (1-indexed) from data/<task>/<dataset>/{train,test}_i.csv.

    Caches ECFP4 features in features/<task>_<dataset>_fold{i}_{train,test}.npz
    to avoid recomputing across notebook re-runs."""
    train_csv = DATA_DIR / f"train_{fold_idx}.csv"
    test_csv  = DATA_DIR / f"test_{fold_idx}.csv"

    train_df = pd.read_csv(train_csv)
    test_df  = pd.read_csv(test_csv)

    cache_dir = FEATURES_DIR / f"{TASK}_{DATASET}_ecfp4"
    cache_dir.mkdir(parents=True, exist_ok=True)
    train_cache = cache_dir / f"fold{fold_idx}_train.npz"
    test_cache  = cache_dir / f"fold{fold_idx}_test.npz"

    if train_cache.exists():
        X_train = np.load(train_cache)["X"]
    else:
        X_train = featurize_split(train_df)
        np.savez_compressed(train_cache, X=X_train)

    if test_cache.exists():
        X_test = np.load(test_cache)["X"]
    else:
        X_test = featurize_split(test_df)
        np.savez_compressed(test_cache, X=X_test)

    # target column: "label" per Hi, "value"/"y"/"target" per Lo (rispetta il tuo schema)
    y_col_candidates = ["label", "y", "target", "value", "pXC50", "activity"]
    y_col = next((c for c in y_col_candidates if c in train_df.columns), train_df.columns[-1])

    y_train = train_df[y_col].values.astype(np.float32)
    y_test  = test_df[y_col].values.astype(np.float32)

    if TASK == "lo":
        # cluster id richiesto per il mean-Spearman intra-cluster
        cluster_col = "cluster" if "cluster" in test_df.columns else "cluster_id"
        clusters_test = test_df[cluster_col].values
    else:
        clusters_test = None

    return X_train, y_train, X_test, y_test, clusters_test


# Sanity check sul fold 1
X_tr, y_tr, X_te, y_te, _ = load_fold(1)
print(f"Fold 1 — X_train: {X_tr.shape}, y_train: {y_tr.shape}  "
      f"(positivi: {int(y_tr.sum())}/{len(y_tr)})  |  X_test: {X_te.shape}")


## 4. Architettura MLP

Costruzione **piramidale** (layer che si restringono): replica l'idea
*hierarchical abstraction* discussa a lezione (Micheli, NN-part3 Deep), dove ogni
layer estrae feature più astratte del precedente.

Dato il numero totale di neuroni `N_total` e il numero di layer `L`, distribuiamo i
neuroni in progressione geometrica con ratio `r` (default 0.5 → piramide).

- `r = 1.0`  → layer di larghezza costante
- `r < 1.0`  → piramide rastremata (compressione progressiva)


In [ ]:
def get_pyramid_sizes(n_input: int, N_total: int, num_layers: int,
                      r: float = 0.5) -> list[int]:
    """Distribute N_total neurons across num_layers in geometric progression.

    Solve a + ar + ar^2 + ... + ar^(L-1) = N_total for the first-layer width 'a'.
    Returns rounded integer sizes (>=8 for safety)."""
    if num_layers == 1:
        return [N_total]
    if abs(r - 1.0) < 1e-9:
        a = N_total / num_layers
    else:
        a = N_total * (1 - r) / (1 - r ** num_layers)
    sizes = [max(8, int(round(a * (r ** k)))) for k in range(num_layers)]
    return sizes


class MLP(nn.Module):
    """Plain feed-forward MLP. Output head depends on task:
       - Hi (binary classification): single logit -> sigmoid in loss
       - Lo (regression):             single linear output"""

    def __init__(self, input_dim: int, hidden_sizes: list[int], task: str,
                 activation: str = "relu", dropout: float = 0.0,
                 use_batchnorm: bool = False, init: str = "kaiming"):
        super().__init__()

        act_map = {
            "relu": nn.ReLU, "leaky_relu": nn.LeakyReLU,
            "tanh": nn.Tanh, "elu": nn.ELU, "gelu": nn.GELU,
        }
        Act = act_map[activation]

        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(Act())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        out_dim = 1   # binary logit (Hi) or scalar regressor (Lo)
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
        self.task = task

        self._init_weights(init, activation)

    def _init_weights(self, init: str, activation: str):
        nonlin = "relu" if activation in ("relu", "leaky_relu", "elu") else "tanh"
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init == "kaiming":
                    nn.init.kaiming_uniform_(m.weight, nonlinearity=nonlin)
                elif init == "xavier":
                    nn.init.xavier_uniform_(m.weight)
                elif init == "kaiming_normal":
                    nn.init.kaiming_normal_(m.weight, nonlinearity=nonlin)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


## 5. Training loop con early stopping

Singola run di training. Restituisce il **modello al miglior epoch di validation**.

Punti chiave:
- **Loss**: `BCEWithLogitsLoss` (Hi) con `pos_weight` per gestire lo sbilanciamento, `MSELoss` (Lo)
- **Early stopping** sulla metrica di validation (PR-AUC per Hi, MSE per Lo)
- **Gradient clipping** a norma 5 (cliff prevention, cfr. Micheli "Clipping gradient")
- **Multiple random init** mediati (riduce varianza dovuta al seed)


In [ ]:
def train_single_run(X_tr, y_tr, X_val, y_val, params, scaler_y=None,
                     verbose=False) -> tuple:
    """Train one MLP and return (best_state, best_val_metric, train_curve, val_curve).

    'best_val_metric' is HIGHER-is-better:
      - Hi: PR-AUC on validation
      - Lo: -MSE on validation (so we always maximize)
    """
    input_dim = X_tr.shape[1]

    # Architecture
    hidden_sizes = get_pyramid_sizes(
        n_input=input_dim,
        N_total=params["n_nodes"],
        num_layers=params["n_layers"],
        r=params["r"],
    )

    model = MLP(
        input_dim=input_dim,
        hidden_sizes=hidden_sizes,
        task=TASK,
        activation=params.get("activation", "relu"),
        dropout=params.get("dropout", 0.0),
        use_batchnorm=params.get("batchnorm", False),
        init=params.get("init", "kaiming"),
    ).to(DEVICE)

    # Optimizer
    opt_name = params.get("optimizer", "adam")
    if opt_name == "adam":
        optimizer = optim.Adam(model.parameters(),
                               lr=params["lr"],
                               weight_decay=params["weight_decay"])
    elif opt_name == "adamw":
        optimizer = optim.AdamW(model.parameters(),
                                lr=params["lr"],
                                weight_decay=params["weight_decay"])
    elif opt_name == "sgd":
        optimizer = optim.SGD(model.parameters(),
                              lr=params["lr"],
                              momentum=params.get("momentum", 0.9),
                              nesterov=True,
                              weight_decay=params["weight_decay"])
    else:
        raise ValueError(opt_name)

    # Loss
    if TASK == "hi":
        n_pos = float(y_tr.sum()); n_neg = len(y_tr) - n_pos
        pos_weight = torch.tensor([max(1.0, n_neg / max(1.0, n_pos))],
                                  device=DEVICE) if params.get("class_weight") == "balanced" \
                     else torch.tensor([1.0], device=DEVICE)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        loss_fn = nn.MSELoss()

    # Tensors / loader
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=DEVICE)
    X_val_t= torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    if TASK == "lo":
        y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)
        y_val_t= torch.tensor(y_val, dtype=torch.float32, device=DEVICE)
    else:
        y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=DEVICE)
        y_val_t= torch.tensor(y_val, dtype=torch.float32, device=DEVICE)

    bs = params.get("batch_size", 256) or len(X_tr_t)
    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)

    # Training loop with early stopping
    best_val = -float("inf")
    best_state = None
    no_improve = 0
    patience = params.get("patience", 20)
    train_curve, val_curve = [], []

    for epoch in range(params["epochs"]):
        # --- TRAIN ---
        model.train()
        epoch_loss = 0.0; n = 0
        for bx, by in train_loader:
            optimizer.zero_grad()
            out = model(bx)
            loss = loss_fn(out, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),
                                           params.get("grad_clip", 5.0))
            optimizer.step()
            epoch_loss += loss.item() * len(bx); n += len(bx)
        train_curve.append(epoch_loss / n)

        # --- VAL ---
        model.eval()
        with torch.no_grad():
            v_out = model(X_val_t)
            if TASK == "hi":
                probs = torch.sigmoid(v_out).cpu().numpy()
                val_metric = average_precision_score(y_val, probs)
            else:
                pred = v_out.cpu().numpy()
                # PR-AUC analog for regression: maximize -MSE
                val_metric = -float(np.mean((pred - y_val) ** 2))
        val_curve.append(val_metric)

        if val_metric > best_val + 1e-5:
            best_val = val_metric
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

        if verbose and epoch % 20 == 0:
            print(f"  epoch {epoch:3d}  train_loss={train_curve[-1]:.4f}  val={val_metric:.4f}")

    return best_state, best_val, train_curve, val_curve


def train_with_inits(X_tr, y_tr, X_val, y_val, params, n_inits: int = 3):
    """Run training n_inits times with different seeds, return median run."""
    runs = []
    for k in range(n_inits):
        torch.manual_seed(SEED + k)
        np.random.seed(SEED + k)
        state, best_val, tc, vc = train_single_run(X_tr, y_tr, X_val, y_val, params)
        runs.append((state, best_val, tc, vc))
    vals = np.array([r[1] for r in runs])
    median_idx = int(np.argsort(vals)[len(vals)//2])
    avg_val = float(np.mean(vals))
    return runs[median_idx], avg_val


## 6. Inner CV: valutazione di una singola configurazione

Ogni configurazione viene valutata su 2 fold inner. Restituisce media ± std della metrica.

In [ ]:
def evaluate_config_inner(X, y, params, n_inner: int = 2,
                          n_inits: int = 2) -> dict:
    """Run inner k-fold CV for a single hyperparameter configuration.

    Returns dict with mean/std of the validation metric across folds."""
    if TASK == "hi":
        cv = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=SEED)
        splitter = cv.split(X, y)
    else:
        cv = KFold(n_splits=n_inner, shuffle=True, random_state=SEED)
        splitter = cv.split(X)

    fold_scores = []
    for tr_idx, val_idx in splitter:
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        # Per Lo: standardizza il target sul training di questo inner-fold
        if TASK == "lo":
            scaler_y = StandardScaler().fit(y_tr.reshape(-1, 1))
            y_tr_s   = scaler_y.transform(y_tr.reshape(-1, 1)).ravel()
            y_val_s  = scaler_y.transform(y_val.reshape(-1, 1)).ravel()
        else:
            y_tr_s, y_val_s = y_tr, y_val

        _, avg_val = train_with_inits(X_tr, y_tr_s, X_val, y_val_s,
                                      params, n_inits=n_inits)
        fold_scores.append(avg_val)

    return {
        "mean": float(np.mean(fold_scores)),
        "std":  float(np.std(fold_scores)),
        "scores_per_fold": [float(s) for s in fold_scores],
    }


## 7. FASE 1 — Random Search

Spazio di ricerca largo. Strategia:
- **profondità** 1–4 layer (con ECFP4 a 2048 bit conviene esplorare anche reti relativamente shallow)
- **n_nodes** totali su scala log
- **lr** loguniform `[1e-4, 1e-2]`
- **weight_decay** loguniform `[1e-6, 1e-2]`
- **dropout** uniforme `[0, 0.5]`
- **batch_size** scelto fra valori canonici

`n_iter=60` è un compromesso ragionevole per coprire lo spazio: con 3 inner-CV
fold e 2 random init parliamo di ~720 train run per dataset, gestibile su CPU
in qualche ora.


In [ ]:
# --- Spazio random search ---
random_space = {
    "n_layers":     [1, 2, 3, 4],
    "n_nodes":      [128, 256, 512, 1024, 2048],
    "r":            [0.5, 0.75, 1.0],
    "activation":   ["relu", "leaky_relu", "elu"],
    "init":         ["kaiming", "kaiming_normal", "xavier"],
    "dropout":      uniform(0.0, 0.5),         # U[0, 0.5]
    "batchnorm":    [False, True],
    "optimizer":    ["adam", "adamw"],
    "lr":           loguniform(1e-4, 1e-2),
    "weight_decay": loguniform(1e-6, 1e-2),
    "batch_size":   [64, 128, 256],
    "epochs":       [200],
    "patience":     [20],
    "grad_clip":    [5.0],
    "class_weight": [None, "balanced"] if TASK == "hi" else [None],
}

N_RANDOM_ITER = 60
N_INNER       = 2     # come nel resto del progetto
N_INITS_INNER = 2     # 2 init durante search per velocità
N_INITS_FINAL = 5     # 5 init nel retrain finale

print(f"Random search: {N_RANDOM_ITER} configs  x  {N_INNER} inner folds  "
      f"x  {N_INITS_INNER} inits = {N_RANDOM_ITER * N_INNER * N_INITS_INNER} train runs / outer fold")


## 8. Esecuzione random search sui 3 outer fold

Per ogni outer fold:
1. Carica `train_i`, `test_i`
2. Sample `N_RANDOM_ITER` config dallo spazio
3. Valuta ogni config con inner CV
4. Sceglie la best config
5. **Retrain** su tutto il train con `N_INITS_FINAL` init  → valuta su test
6. Salva tutto in JSON (stesso formato del resto del progetto)


In [ ]:
def compute_test_metric(model_state, X_train, y_train, X_test, y_test,
                        clusters_test, params) -> dict:
    """Build a fresh model, load the state, predict, return metrics."""
    input_dim = X_train.shape[1]
    hidden_sizes = get_pyramid_sizes(input_dim, params["n_nodes"],
                                     params["n_layers"], params["r"])
    model = MLP(input_dim, hidden_sizes, TASK,
                activation=params.get("activation", "relu"),
                dropout=params.get("dropout", 0.0),
                use_batchnorm=params.get("batchnorm", False),
                init=params.get("init", "kaiming")).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    with torch.no_grad():
        X_te_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
        out = model(X_te_t).cpu().numpy()

    metrics = {}
    if TASK == "hi":
        probs = 1 / (1 + np.exp(-out))
        metrics["pr_auc"] = float(average_precision_score(y_test, probs))
    else:
        # mean Spearman intra-cluster
        df = pd.DataFrame({"y": y_test, "pred": out, "cluster": clusters_test})
        rhos = []
        for c, g in df.groupby("cluster"):
            if len(g) >= 2 and g["y"].nunique() > 1:
                rho, _ = spearmanr(g["y"], g["pred"])
                if not np.isnan(rho):
                    rhos.append(rho)
        metrics["mean_spearman"] = float(np.mean(rhos)) if rhos else 0.0
        metrics["n_clusters_used"] = len(rhos)
    return metrics


def retrain_on_full_train(X_train, y_train, X_test, y_test, clusters_test,
                          best_params, n_inits: int = 5) -> dict:
    """Retrain on full training set with multiple inits, average predictions
    for stability. Save mean ensemble metric."""
    if TASK == "lo":
        scaler_y = StandardScaler().fit(y_train.reshape(-1, 1))
        y_train_s = scaler_y.transform(y_train.reshape(-1, 1)).ravel()
        y_test_s  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()
    else:
        y_train_s, y_test_s = y_train, y_test

    # Use 10% of train as internal validation for early stopping at retrain time
    n = len(X_train); n_val = max(64, int(0.1 * n))
    rng = np.random.RandomState(SEED)
    perm = rng.permutation(n)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    X_tr2, y_tr2 = X_train[tr_idx], y_train_s[tr_idx]
    X_va2, y_va2 = X_train[val_idx], y_train_s[val_idx]

    test_outs = []
    for k in range(n_inits):
        torch.manual_seed(SEED + 100 + k)
        np.random.seed(SEED + 100 + k)
        best_state, _, _, _ = train_single_run(X_tr2, y_tr2, X_va2, y_va2,
                                               best_params)
        # Predict on test
        input_dim = X_train.shape[1]
        hs = get_pyramid_sizes(input_dim, best_params["n_nodes"],
                               best_params["n_layers"], best_params["r"])
        m = MLP(input_dim, hs, TASK,
                activation=best_params.get("activation", "relu"),
                dropout=best_params.get("dropout", 0.0),
                use_batchnorm=best_params.get("batchnorm", False),
                init=best_params.get("init", "kaiming")).to(DEVICE)
        m.load_state_dict(best_state); m.eval()
        with torch.no_grad():
            X_te_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)
            test_outs.append(m(X_te_t).cpu().numpy())

    # Ensemble = mean of init-runs
    test_ens = np.mean(test_outs, axis=0)

    metrics = {}
    if TASK == "hi":
        probs = 1 / (1 + np.exp(-test_ens))
        metrics["pr_auc"] = float(average_precision_score(y_test, probs))
    else:
        # invert scaling
        pred_real = scaler_y.inverse_transform(test_ens.reshape(-1, 1)).ravel()
        df = pd.DataFrame({"y": y_test, "pred": pred_real, "cluster": clusters_test})
        rhos = []
        for c, g in df.groupby("cluster"):
            if len(g) >= 2 and g["y"].nunique() > 1:
                rho, _ = spearmanr(g["y"], g["pred"])
                if not np.isnan(rho):
                    rhos.append(rho)
        metrics["mean_spearman"]   = float(np.mean(rhos)) if rhos else 0.0
        metrics["n_clusters_used"] = len(rhos)
    return metrics


In [ ]:
def run_search_on_outer_folds(search_iter, search_kind: str = "random",
                              random_space: dict = None,
                              grid_space: dict = None,
                              save_dir: Path = None) -> list:
    """Run nested CV across the 3 outer folds.

    search_kind: 'random' or 'grid'.
    Saves params_fold_{i}.json and the full search log."""

    save_dir = save_dir or RESULTS_DIR
    save_dir.mkdir(parents=True, exist_ok=True)
    all_fold_results = []

    for outer in (1, 2, 3):
        print(f"\n========================================")
        print(f"  OUTER FOLD {outer}/3 — {search_kind.upper()} SEARCH")
        print(f"========================================")

        X_train, y_train, X_test, y_test, clusters_test = load_fold(outer)

        # Build sampler
        if search_kind == "random":
            sampler = ParameterSampler(random_space, n_iter=search_iter,
                                       random_state=SEED + outer)
        else:
            sampler = ParameterGrid(grid_space)

        config_list = list(sampler)
        print(f"  Configs to evaluate: {len(config_list)}")

        log_records = []
        t0 = time.time()
        for i, params in enumerate(config_list, 1):
            t_cfg = time.time()
            try:
                res = evaluate_config_inner(X_train, y_train, params,
                                            n_inner=N_INNER,
                                            n_inits=N_INITS_INNER)
            except Exception as e:
                print(f"    [{i}/{len(config_list)}] FAILED: {e}")
                continue
            log_records.append({"params": params, "inner": res,
                                "time_sec": time.time() - t_cfg})
            if i % 5 == 0 or i == len(config_list):
                elapsed = time.time() - t0
                eta = elapsed / i * (len(config_list) - i)
                print(f"    [{i:3d}/{len(config_list)}]  "
                      f"inner={res['mean']:.4f}±{res['std']:.4f}  "
                      f"elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")

        # Pick best
        log_records.sort(key=lambda r: -r["inner"]["mean"])
        best = log_records[0]
        print(f"\n  Best inner score: {best['inner']['mean']:.4f}")
        print(f"  Best params: {best['params']}")

        # Retrain on full training set + evaluate on test
        print(f"  Retraining with {N_INITS_FINAL} inits on full train...")
        test_metrics = retrain_on_full_train(X_train, y_train,
                                             X_test, y_test, clusters_test,
                                             best["params"],
                                             n_inits=N_INITS_FINAL)
        print(f"  Outer test: {test_metrics}")

        # Save params_fold_{outer}.json (stesso schema del resto del progetto)
        fold_blob = {
            "fold":          outer,
            "model":         MODEL_NAME,
            "task":          TASK,
            "dataset":       DATASET,
            "search_kind":   search_kind,
            "best_params":   best["params"],
            "inner_cv":      best["inner"],
            "test_metrics":  test_metrics,
            "n_configs_evaluated": len(log_records),
            "timestamp":     datetime.datetime.now().isoformat(),
        }
        with open(save_dir / f"params_fold_{outer}.json", "w") as f:
            json.dump(fold_blob, f, indent=2, default=str)

        # Save full search log
        with open(save_dir / f"search_log_{search_kind}_fold_{outer}.json", "w") as f:
            json.dump(log_records, f, indent=2, default=str)

        all_fold_results.append(fold_blob)

    return all_fold_results


In [ ]:
# === ESEGUI RANDOM SEARCH ===
# (Commenta questa cella e passa alla grid se hai già fatto il random)

random_save = RESULTS_DIR / "random_search"
random_results = run_search_on_outer_folds(
    search_iter   = N_RANDOM_ITER,
    search_kind   = "random",
    random_space  = random_space,
    save_dir      = random_save,
)


## 9. Analisi risultati random search

Aggreghiamo i 3 fold e guardiamo:
- Score medio finale (á la `mean ± std` del progetto)
- Quali iperparametri ricorrono nei top-k della search log → informa la grid


In [ ]:
def summarize_results(fold_results, metric_name=None):
    """Mean ± std on the outer folds, like the rest of the project tables."""
    if metric_name is None:
        metric_name = "pr_auc" if TASK == "hi" else "mean_spearman"
    scores = [r["test_metrics"][metric_name] for r in fold_results]
    mu, sd = float(np.mean(scores)), float(np.std(scores))
    print(f"\n=== {MODEL_NAME} on {DATASET}-{TASK} ({metric_name}) ===")
    for r, s in zip(fold_results, scores):
        print(f"  fold {r['fold']}: {s:.4f}")
    print(f"  AGGREGATE: {mu:.4f} ± {sd:.4f}")
    return mu, sd, scores

mu_random, sd_random, scores_random = summarize_results(random_results)


In [ ]:
# Top-k analysis: che iperparametri compaiono nelle config migliori across folds?
from collections import Counter

def topk_param_frequency(save_dir: Path, k: int = 10):
    counters = {}
    all_top = []
    for fold in (1, 2, 3):
        with open(save_dir / f"search_log_random_fold_{fold}.json") as f:
            log = json.load(f)
        log.sort(key=lambda r: -r["inner"]["mean"])
        all_top.extend([r["params"] for r in log[:k]])

    keys_to_inspect = ["n_layers", "n_nodes", "r", "activation", "init",
                       "batchnorm", "optimizer", "batch_size", "class_weight"]
    for key in keys_to_inspect:
        vals = [str(p.get(key)) for p in all_top]
        counters[key] = Counter(vals).most_common()
    return counters, all_top

cnt, all_top = topk_param_frequency(random_save, k=10)
print(f"Top-10 configs across 3 folds  ({len(all_top)} total):\n")
for k, v in cnt.items():
    print(f"  {k:14s}  ->  {v}")

# Continuous params — show median & IQR for the top configs
import statistics as st
for key in ["lr", "weight_decay", "dropout"]:
    vals = [p[key] for p in all_top if key in p]
    if vals:
        vals_sorted = sorted(vals)
        q1 = vals_sorted[len(vals)//4]
        med = st.median(vals)
        q3 = vals_sorted[3*len(vals)//4]
        print(f"  {key:14s}  ->  median={med:.2e}   IQR=[{q1:.2e}, {q3:.2e}]")


## 10. FASE 2 — Grid Search raffinata

Template di grid — **DA RIEMPIRE** dopo aver letto l'output della cella precedente.

Idea: prendi i parametri più frequenti nei top-10 e le mediane delle distribuzioni
continue, e costruisci una grid stretta intorno. Esempio (è solo una traccia, **modifica
i valori in base a quello che vedi**):

```python
grid_space = {
    "n_layers":     [<top-1>, <top-2>],
    "n_nodes":      [<top>, <top> * 2],
    "r":            [<top>],
    "activation":   ["relu"],
    "init":         ["kaiming"],
    "dropout":      [median - 0.05, median, median + 0.1],
    "batchnorm":    [<top>],
    "optimizer":    ["adamw"],
    "lr":           [median * 0.5, median, median * 2],
    "weight_decay": [median, median * 10],
    ...
}
```

Tieni la grid sotto le ~80 combinazioni totali per restare nei tempi.


In [ ]:
# === GRID SEARCH RAFFINATA ===
# >>> RIEMPIRE QUESTI VALORI dopo aver visto la random search <<<

grid_space = {
    "n_layers":     [2, 3],                    # TODO: dai top
    "n_nodes":      [512, 1024],               # TODO
    "r":            [0.5, 0.75],               # TODO
    "activation":   ["relu"],                  # TODO
    "init":         ["kaiming"],               # TODO
    "dropout":      [0.1, 0.2, 0.3],           # TODO: intorno alla mediana
    "batchnorm":    [False],                   # TODO
    "optimizer":    ["adamw"],                 # TODO
    "lr":           [5e-4, 1e-3, 2e-3],        # TODO
    "weight_decay": [1e-4, 1e-3],              # TODO
    "batch_size":   [128],                     # TODO
    "epochs":       [300],                     # un po' più lungo
    "patience":     [30],
    "grad_clip":    [5.0],
    "class_weight": ["balanced"] if TASK == "hi" else [None],
}

# Sanity check: quante combinazioni?
n_combos = 1
for v in grid_space.values():
    n_combos *= len(v)
print(f"Grid combinations: {n_combos}  (advised: < 80)")


In [ ]:
# === ESEGUI GRID SEARCH ===
grid_save = RESULTS_DIR / "grid_search"
grid_results = run_search_on_outer_folds(
    search_iter  = None,
    search_kind  = "grid",
    grid_space   = grid_space,
    save_dir     = grid_save,
)

mu_grid, sd_grid, scores_grid = summarize_results(grid_results)


## 11. Risultati finali — scelta migliore tra random e grid

Per ciascun outer fold confronto le due ricerche e tengo la migliore. Il risultato finale
(quello che andrà in tabella tesi) viene salvato in `results/{task}/{dataset}/mlp_{dataset}_ecfp4/params_fold_{i}.json`,
**stesso schema del resto del progetto**, così funzionano gli script di aggregazione già esistenti.


In [ ]:
def pick_best_per_fold(random_save: Path, grid_save: Path,
                       final_dir: Path) -> list:
    metric_name = "pr_auc" if TASK == "hi" else "mean_spearman"
    final_dir.mkdir(parents=True, exist_ok=True)
    finals = []
    for fold in (1, 2, 3):
        with open(random_save / f"params_fold_{fold}.json") as f:
            r = json.load(f)
        try:
            with open(grid_save / f"params_fold_{fold}.json") as f:
                g = json.load(f)
        except FileNotFoundError:
            g = None

        if g is None or r["test_metrics"][metric_name] >= g["test_metrics"][metric_name]:
            chosen, src = r, "random"
        else:
            chosen, src = g, "grid"
        chosen["selected_via"] = src
        finals.append(chosen)
        with open(final_dir / f"params_fold_{fold}.json", "w") as f:
            json.dump(chosen, f, indent=2, default=str)
        print(f"  fold {fold}: {src:>6s}  score={chosen['test_metrics'][metric_name]:.4f}")
    return finals

print("Selecting best run per fold...")
finals = pick_best_per_fold(random_save, grid_save, RESULTS_DIR)

print()
mu_final, sd_final, scores_final = summarize_results(finals)
print()
print(f">>> FINAL RESULT for {MODEL_NAME} on {DATASET}-{TASK}: "
      f"{mu_final:.4f} ± {sd_final:.4f} <<<")


In [ ]:
# Save a small aggregate summary, easy to grep across datasets later
summary = {
    "model":   MODEL_NAME,
    "task":    TASK,
    "dataset": DATASET,
    "metric":  "pr_auc" if TASK == "hi" else "mean_spearman",
    "scores_per_fold": scores_final,
    "mean":    mu_final,
    "std":     sd_final,
    "fingerprint": "ecfp4",
    "timestamp":  datetime.datetime.now().isoformat(),
}
with open(RESULTS_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print("Saved:", RESULTS_DIR / "summary.json")
